# Import dependencies

In [3]:
import pandas as pd
import numpy as np

# Data

## Load resale data

In [4]:
resale = pd.read_csv("Data\\resale_with_cords.csv")

In [5]:
print(f"Shape: {resale.shape}")
print(resale.info())
resale.describe(include='all').T

Shape: (108, 14)
<class 'pandas.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   month                108 non-null    str    
 1   town                 108 non-null    str    
 2   flat_type            108 non-null    str    
 3   block                108 non-null    str    
 4   street_name          108 non-null    str    
 5   storey_range         108 non-null    str    
 6   floor_area_sqm       108 non-null    int64  
 7   flat_model           108 non-null    str    
 8   lease_commence_date  108 non-null    int64  
 9   remaining_lease      108 non-null    str    
 10  resale_price         108 non-null    int64  
 11  address              108 non-null    str    
 12  latitude             108 non-null    float64
 13  longitude            108 non-null    float64
dtypes: float64(2), int64(3), str(9)
memory usage: 11.9 KB
None


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
month,108,1,2017-01,108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
town,108,2,ANG MO KIO,56,NaN,NaN,NaN,NaN,NaN,NaN,NaN
flat_type,108,5,3 ROOM,63,NaN,NaN,NaN,NaN,NaN,NaN,NaN
block,108,92,709,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
street_name,108,22,ANG MO KIO AVE 10,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
storey_range,108,7,04 TO 06,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
floor_area_sqm,108.0,NaN,NaN,NaN,79.12037,17.035267,44.0,67.0,73.0,91.0,147.0
flat_model,108,7,New Generation,72,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lease_commence_date,108.0,NaN,NaN,NaN,1981.472222,7.363028,1974.0,1978.0,1980.0,1981.0,2012.0
remaining_lease,108,55,62 years 05 months,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Missing Data Check [None]

In [6]:
pd.DataFrame({
    'nunique': resale.nunique(dropna=False),
    'missing': resale.isna().sum(),
}).sort_values('nunique', ascending=False)

,nunique,missing
longitude,97,0
latitude,97,0
address,97,0
block,92,0
resale_price,70,0
remaining_lease,55,0
floor_area_sqm,31,0
street_name,22,0
lease_commence_date,17,0
storey_range,7,0


## Data Description

In [7]:
# 'role': 'feature', 'target', 'identifier', 'metadata', 'ambiguous'
# 'type':  'num-cont', 'num-disc', 'cat-nom', 'cat-ord', 'bool'
role_map = {
    "month" : "feature",
    "town": "feature",
    "flat_type": "feature",
    "block": "identifier",
    "street_name": "identifier",
    "storey_range": "feature",
    "floor_area_sqm": "feature",
    "flat_model": "metadata",
    "lease_commence_date": "metadata",
    "remaining_lease": "feature",
    "resale_price": "target" 
}
type_map = {
    "month" : "num-disc",
    "town": "cat-nom",
    "flat_type": "cat-nom",
    "block": "cat-nom",
    "street_name": "cat-nom",
    "storey_range": "cat-ord",
    "floor_area_sqm": "num-disc",
    "flat_model": "cat-nom",
    "lease_commence_date": "num-disc",
    "remaining_lease": "num-disc",
    "resale_price": "num-disc" 
}

data_description = pd.DataFrame({
    "column": resale.columns,
})
data_description["role"] = data_description["column"].map(role_map).fillna("unknown")
data_description["type"] = data_description["column"].map(type_map).fillna("unknown")
data_description

,column,role,type
0,month,feature,num-disc
1,town,feature,cat-nom
2,flat_type,feature,cat-nom
3,block,identifier,cat-nom
4,street_name,identifier,cat-nom
5,storey_range,feature,cat-ord
6,floor_area_sqm,feature,num-disc
7,flat_model,metadata,cat-nom
8,lease_commence_date,metadata,num-disc
9,remaining_lease,feature,num-disc


## Data Pre-Processing

### Model Dataset Descriptions

In [8]:
model_data_desc = data_description[data_description['role'] != "metadata"]
model_data_desc

,column,role,type
0,month,feature,num-disc
1,town,feature,cat-nom
2,flat_type,feature,cat-nom
3,block,identifier,cat-nom
4,street_name,identifier,cat-nom
5,storey_range,feature,cat-ord
6,floor_area_sqm,feature,num-disc
9,remaining_lease,feature,num-disc
10,resale_price,target,num-disc
11,address,unknown,unknown


### Model Dataset

In [9]:
model_data = resale[model_data_desc["column"]].copy()
model_data.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44,61 years 04 months,232000,406 ANG MO KIO AVE 10,1.362005,103.853880
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67,60 years 07 months,250000,108 ANG MO KIO AVE 4,1.370966,103.838202
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67,62 years 05 months,262000,602 ANG MO KIO AVE 5,1.380709,103.835368
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68,62 years 01 month,265000,465 ANG MO KIO AVE 10,1.366201,103.857201
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67,62 years 05 months,265000,601 ANG MO KIO AVE 5,1.381041,103.835132


### Data Transformation

**Month** \
Convert to indexing for model input, index represents the order for time-series models \
index starts from 2017-01 onwards \
Example:

| Month(before) | index(after) |
| ------ | ------ |
| 2017-01 | 0 |
| 2017-02 | 1 |

In [10]:
months = pd.to_datetime(model_data['month'], format='%Y-%m')
model_data['month'] = (
    months.dt.year * 12 + months.dt.month
)

model_data['month'] -= model_data['month'].min()
model_data.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44,61 years 04 months,232000,406 ANG MO KIO AVE 10,1.362005,103.853880
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67,60 years 07 months,250000,108 ANG MO KIO AVE 4,1.370966,103.838202
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67,62 years 05 months,262000,602 ANG MO KIO AVE 5,1.380709,103.835368
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68,62 years 01 month,265000,465 ANG MO KIO AVE 10,1.366201,103.857201
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67,62 years 05 months,265000,601 ANG MO KIO AVE 5,1.381041,103.835132


**remaining_lease [Months]** \
Convert remaining_lease to number of months instead of X years X months

In [11]:
remaining_yrs = model_data['remaining_lease'].str.extract(r'(\d+)\D+(?:(\d+)\D+)?').fillna(0).astype(int)
model_data['remaining_lease'] = remaining_yrs[0] * 12 + remaining_yrs[1]
model_data.head()

,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44,736,232000,406 ANG MO KIO AVE 10,1.362005,103.853880
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67,727,250000,108 ANG MO KIO AVE 4,1.370966,103.838202
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67,749,262000,602 ANG MO KIO AVE 5,1.380709,103.835368
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68,745,265000,465 ANG MO KIO AVE 10,1.366201,103.857201
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67,749,265000,601 ANG MO KIO AVE 5,1.381041,103.835132


**Storey type** \
Replace storey range with the storey type (lower, middle, upper)

In [12]:
avg_storey = model_data['storey_range'].str.split(" TO ", expand=True).astype(int).mean(axis=1)
model_data['storey_type'] = pd.cut(avg_storey, bins=[1,3, 7,99], labels=['lower','middle','upper'], right=False)
model_data = pd.get_dummies(model_data, columns=['storey_type'])
model_data.drop(columns=['storey_range'], inplace=True)
model_data.head()

,month,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,44,736,232000,406 ANG MO KIO AVE 10,1.362005,103.853880,False,False,True
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,67,727,250000,108 ANG MO KIO AVE 4,1.370966,103.838202,True,False,False
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,67,749,262000,602 ANG MO KIO AVE 5,1.380709,103.835368,True,False,False
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,68,745,265000,465 ANG MO KIO AVE 10,1.366201,103.857201,False,True,False
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,67,749,265000,601 ANG MO KIO AVE 5,1.381041,103.835132,True,False,False


**Quarter index** \
Defines which quarter is the data in example 2 for 2017-04 to 2017-07

In [13]:
model_data['quarter'] = model_data['month'] // 3
model_data.tail()

,month,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,quarter
103,0,BEDOK,4 ROOM,554,BEDOK NTH ST 3,93,749,412000,554 BEDOK NTH ST 3,1.332627,103.928172,False,True,False,0
104,0,BEDOK,4 ROOM,602,BEDOK RESERVOIR RD,98,772,420000,602 BEDOK RESERVOIR RD,1.329349,103.911583,False,False,True,0
105,0,BEDOK,4 ROOM,81,BEDOK NTH RD,91,725,425000,81 BEDOK NTH RD,1.328916,103.940529,False,False,True,0
106,0,BEDOK,4 ROOM,508,BEDOK NTH AVE 3,92,730,430000,508 BEDOK NTH AVE 3,1.333400,103.932519,False,False,True,0
107,0,BEDOK,4 ROOM,720,BEDOK RESERVOIR RD,104,793,430000,720 BEDOK RESERVOIR RD,1.335973,103.924883,False,True,False,0


**Resale Price Index [RPI]**

In [14]:
rpi = pd.read_csv('Data\\2025-RPI.csv')
rpi = rpi[rpi['year'] >= 2017]
rpi['quarter'] = (rpi['year'] - 2017) * 4 + (rpi['quarter'] - 1)
rpi.drop(columns=['year'], inplace=True)
rpi.head()

,quarter,rpi
32,0,133.9
33,1,133.7
34,2,132.8
35,3,132.6
36,4,131.6


In [15]:
model_data = pd.merge(model_data, rpi, on='quarter',how='left')
model_data.head()

,month,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,quarter,rpi
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,44,736,232000,406 ANG MO KIO AVE 10,1.362005,103.853880,False,False,True,0,133.9
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,67,727,250000,108 ANG MO KIO AVE 4,1.370966,103.838202,True,False,False,0,133.9
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,67,749,262000,602 ANG MO KIO AVE 5,1.380709,103.835368,True,False,False,0,133.9
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,68,745,265000,465 ANG MO KIO AVE 10,1.366201,103.857201,False,True,False,0,133.9
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,67,749,265000,601 ANG MO KIO AVE 5,1.381041,103.835132,True,False,False,0,133.9


**Adjusted Resale Price** \
Formula: Resale price / RPI

In [16]:
model_data['adjusted_resale'] = model_data['resale_price'] / (model_data['rpi'] / 100)
model_data.head()

,month,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,address,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,quarter,rpi,adjusted_resale
0,0,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,44,736,232000,406 ANG MO KIO AVE 10,1.362005,103.853880,False,False,True,0,133.9,173263.629574
1,0,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,67,727,250000,108 ANG MO KIO AVE 4,1.370966,103.838202,True,False,False,0,133.9,186706.497386
2,0,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,67,749,262000,602 ANG MO KIO AVE 5,1.380709,103.835368,True,False,False,0,133.9,195668.409261
3,0,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,68,745,265000,465 ANG MO KIO AVE 10,1.366201,103.857201,False,True,False,0,133.9,197908.887229
4,0,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,67,749,265000,601 ANG MO KIO AVE 5,1.381041,103.835132,True,False,False,0,133.9,197908.887229


**Remove unneccessary columns**

In [17]:
transformed = model_data.drop(columns=['block','street_name','quarter','address'])
transformed.head()

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,rpi,adjusted_resale
0,0,ANG MO KIO,2 ROOM,44,736,232000,1.362005,103.853880,False,False,True,133.9,173263.629574
1,0,ANG MO KIO,3 ROOM,67,727,250000,1.370966,103.838202,True,False,False,133.9,186706.497386
2,0,ANG MO KIO,3 ROOM,67,749,262000,1.380709,103.835368,True,False,False,133.9,195668.409261
3,0,ANG MO KIO,3 ROOM,68,745,265000,1.366201,103.857201,False,True,False,133.9,197908.887229
4,0,ANG MO KIO,3 ROOM,67,749,265000,1.381041,103.835132,True,False,False,133.9,197908.887229


### Data Aggregation [KIV]

The dataset contains resale price data for specific blocks and streets which is too specific for model training. Hence, we are aggregating the resale price by the following features instead:

- town
- flat_type
- floor_area_sqm
- remaining_lease
- avg_storey
- month [index]

Data uniqness

In [36]:
model_data[(model_data['block'] == "116") & (model_data['town'] == "ANG MO KIO")].head()

,town,flat_type,block,street_name,floor_area_sqm,remaining_lease,resale_price,storey_type_lower,storey_type_middle,storey_type_upper,quarter
18925,ANG MO KIO,2 ROOM,116,ANG MO KIO AVE 4,44.0,713,222000.0,False,False,True,3
30198,ANG MO KIO,2 ROOM,116,ANG MO KIO AVE 4,44.0,706,202000.0,False,False,True,6
36789,ANG MO KIO,2 ROOM,116,ANG MO KIO AVE 4,44.0,703,185000.0,True,False,False,7
58293,ANG MO KIO,2 ROOM,116,ANG MO KIO AVE 4,44.0,691,187000.0,False,True,False,11
66167,ANG MO KIO,2 ROOM,116,ANG MO KIO AVE 4,44.0,687,192000.0,True,False,False,12


In [ ]:
agg_data = model_data.copy()
agg_data = agg_data.groupby(["town","flat_type","block","street_name","floor_area_sqm", "remaining_lease","storey_type_lower","storey_type_middle","storey_type_upper","quarter"])
agg_data.agg({'resale_price':'mean',
              "town":"first",
              "flat_type":"first",
              "block":"first",
              "street_name":"first",
              "floor_area_sqm":"first",
              "remai"
              }).head()

resale_price
town       flat_type block street_name      floor_area_sqm remaining_lease storey_type_lower storey_type_middle storey_type_upper quarter              
ANG MO KIO 2 ROOM    116   ANG MO KIO AVE 4 44.0           630             False             True               False             31           288000.0
                                                           631             False             False              True              31           310000.0
                                                           642             False             False              True              28           288000.0
                                                           643             False             False              True              27           278000.0
                                                           658             True              False              False             22           218888.0

#### Stats

Months per group

In [39]:
agg_data.groupby(["town","flat_type","remaining_lease","storey_type_lower","block","street_name","storey_type_middle","storey_type_upper"])["quarter"].nunique().describe()

count    210289.000000
mean          1.008260
std           0.090509
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           2.000000
Name: quarter, dtype: float64

In [36]:
agg_data = agg_data.sort_values(["town","flat_type","remaining_lease","storey_type_lower","block","street_name","storey_type_middle","storey_type_upper"])

agg_data['month_diff'] = (
    agg_data.groupby(["town","flat_type","remaining_lease","storey_type_lower","block","street_name","storey_type_middle","storey_type_upper"])["month"].diff()
)

agg_data['month_diff'].value_counts()

month_diff
 0.0     9605
 1.0     4955
 2.0      220
 3.0       41
 4.0        3
 10.0       2
 12.0       2
 6.0        2
-1.0        2
 7.0        2
 9.0        2
-2.0        1
 8.0        1
Name: count, dtype: int64

**Town** \
Change to one-hot encoding 

In [80]:
model_data = pd.get_dummies(model_data, columns=["town"], drop_first=True)
model_data.head()

,month,flat_type,block,street_name,ppsm,town_BEDOK,town_BISHAN,town_BUKIT BATOK,town_BUKIT MERAH,town_BUKIT PANJANG,...,town_PASIR RIS,town_PUNGGOL,town_QUEENSTOWN,town_SEMBAWANG,town_SENGKANG,town_SERANGOON,town_TAMPINES,town_TOA PAYOH,town_WOODLANDS,town_YISHUN
0,0,2 ROOM,406,ANG MO KIO AVE 10,7.16,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,0,3 ROOM,108,ANG MO KIO AVE 4,5.13,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,0,3 ROOM,602,ANG MO KIO AVE 5,5.22,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,0,3 ROOM,465,ANG MO KIO AVE 10,5.23,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,0,3 ROOM,601,ANG MO KIO AVE 5,5.28,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


# Feature Extraction

## Import Dependencies

In [1]:
import geopandas as gpd
from shapely.geometry import Point

## Load Data

In [27]:
mrt_df = pd.read_csv("Data\\mrt_stations.csv")

## Project Data on Point

In [33]:
mrt_gdf = gpd.GeoDataFrame(mrt_df, geometry=gpd.points_from_xy(mrt_df.long, mrt_df.lat), crs="EPSG:4326")
mrt_gdf.head()

,station_name,lat,long,geometry
0,Admiralty,1.440637,103.800959,POINT (103.80096 1.44064)
1,Aljunied,1.316452,103.882909,POINT (103.88291 1.31645)
2,Ang Mo Kio,1.370080,103.849523,POINT (103.84952 1.37008)
3,Aviation Park,1.370172,104.003517,POINT (104.00352 1.37017)
4,Bahar Junction,1.345810,103.702356,POINT (103.70236 1.34581)


In [34]:
flat_gdf = gpd.GeoDataFrame(transformed, geometry=gpd.points_from_xy(transformed.longitude, transformed.latitude), crs="EPSG:4326")
flat_gdf.head()

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,rpi,adjusted_resale,mrt_count,geometry
0,0,ANG MO KIO,2 ROOM,44,736,232000,1.362005,103.853880,False,False,True,133.9,173263.629574,8,POINT (103.85388 1.362)
1,0,ANG MO KIO,3 ROOM,67,727,250000,1.370966,103.838202,True,False,False,133.9,186706.497386,7,POINT (103.8382 1.37097)
2,0,ANG MO KIO,3 ROOM,67,749,262000,1.380709,103.835368,True,False,False,133.9,195668.409261,6,POINT (103.83537 1.38071)
3,0,ANG MO KIO,3 ROOM,68,745,265000,1.366201,103.857201,False,True,False,133.9,197908.887229,7,POINT (103.8572 1.3662)
4,0,ANG MO KIO,3 ROOM,67,749,265000,1.381041,103.835132,True,False,False,133.9,197908.887229,6,POINT (103.83513 1.38104)


In [35]:
# Reproject to a Metric CRS (Singapore SVY21 is EPSG:3414)
flat_gdf = flat_gdf.to_crs(epsg=3414)
mrt_gdf = mrt_gdf.to_crs(epsg=3414)

## Get Nearby amenties in radius (500M)

In [36]:
# Create a 500m buffer around each house
flat_gdf['geometry'] = flat_gdf.geometry.buffer(500)

# Spatial Join: Count MRT stations inside the house buffers
joined = gpd.sjoin(flat_gdf, mrt_gdf, how="left", predicate="intersects")

mrt_counts = joined.groupby(joined.index).size() - joined['index_right'].isna().groupby(joined.index).sum()
transformed['mrt_count'] = mrt_counts.values
transformed.head()

,month,town,flat_type,floor_area_sqm,remaining_lease,resale_price,latitude,longitude,storey_type_lower,storey_type_middle,storey_type_upper,rpi,adjusted_resale,mrt_count
0,0,ANG MO KIO,2 ROOM,44,736,232000,1.362005,103.853880,False,False,True,133.9,173263.629574,0
1,0,ANG MO KIO,3 ROOM,67,727,250000,1.370966,103.838202,True,False,False,133.9,186706.497386,1
2,0,ANG MO KIO,3 ROOM,67,749,262000,1.380709,103.835368,True,False,False,133.9,195668.409261,1
3,0,ANG MO KIO,3 ROOM,68,745,265000,1.366201,103.857201,False,True,False,133.9,197908.887229,0
4,0,ANG MO KIO,3 ROOM,67,749,265000,1.381041,103.835132,True,False,False,133.9,197908.887229,1


# Export Data

In [76]:
transformed.to_csv('Data\\processed.csv', index=False)